In [ ]:
"""
Main Pipeline
1. Load sensor datasets
2. Remove outliers using IQR filtering
3. Aggregate minute-level data into daily features
4. Perform feature selection using correlation analysis
5. Train XGBoost regression model with K-Fold validation
6. Predict growth rate for test cases
"""

import pandas as pd
import numpy as np
import glob
import statistics
import xgboost as xgb

from sklearn.model_selection import KFold

In [ ]:
# --------------------------------------------------
# Data Paths
# --------------------------------------------------

train_input_list = sorted(glob.glob("data/train_input/*.csv"))
train_target_list = sorted(glob.glob("data/train_target/*.csv"))

test_input_list = sorted(glob.glob("data/test_input/*.csv"))
test_target_list = sorted(glob.glob("data/test_target/*.csv"))

In [ ]:
# --------------------------------------------------
# Outlier Handling
# --------------------------------------------------

def remove_outliers_iqr(series):
    """
    Replace extreme values using IQR rule.

    Environmental sensor data often contains noise or spikes.
    This function replaces outliers with the median value.
    """
    q3 = series.quantile(0.75)
    q1 = series.quantile(0.25)

    iqr = q3 - q1
    med = statistics.median(series)

    series = series.where(series < (q3 + 1.5 * iqr), med)
    series = series.where(series > (q1 - 1.5 * iqr), med)

    return series

In [ ]:
# --------------------------------------------------
# Load Target Data
# --------------------------------------------------

target_df = pd.DataFrame()
target_lengths = []

for path in train_target_list:

    target = pd.read_csv(path)

    # remove outliers
    target["rate"] = remove_outliers_iqr(target["rate"])

    # temporal smoothing
    target["rate"] = target["rate"].ewm(span=3).mean()

    target_lengths.append(len(target))
    target_df = pd.concat([target_df, target["rate"]], ignore_index=False)

In [ ]:
# --------------------------------------------------
# Feature Extraction (Minute → Daily)
# --------------------------------------------------

first_df = pd.read_csv(train_input_list[0])
columns = first_df.columns

daily_features = pd.DataFrame()

for path in train_input_list:

    data = pd.read_csv(path)
    data.columns = columns

    total_days = int(len(data)) // 1440

    day = 1

    while day <= total_days:

        # Extract one day of minute-level data
        minute_data = data.iloc[1440*(day-1):1440*day]

        feature_vector = []

        for col in columns[1:]:

            # remove outliers for continuous variables
            if (
                "상태" not in col
                and "남은시간" not in col
                and "분무량" not in col
            ):
                minute_data[col] = remove_outliers_iqr(minute_data[col])

            # daily aggregation (mean)
            feature_vector.append(minute_data[col].mean())

        daily_features = pd.concat(
            [daily_features, pd.DataFrame([feature_vector])],
            ignore_index=True
        )

        day += 1


daily_features.columns = columns[1:]

In [ ]:
daily_features.isna().sum()

내부온도관측치          0
내부습도관측치          0
CO2관측치           0
EC관측치            0
외부온도관측치          0
외부습도관측치          0
펌프상태             0
펌프작동남은시간         0
최근분무량            0
일간누적분무량          0
냉방상태             0
냉방작동남은시간         0
난방상태             0
난방작동남은시간         0
내부유동팬상태          0
내부유동팬작동남은시간      0
외부환기팬상태          0
외부환기팬작동남은시간      0
화이트 LED상태        0
화이트 LED작동남은시간    0
화이트 LED동작강도      0
레드 LED상태         0
레드 LED작동남은시간     0
레드 LED동작강도       0
블루 LED상태         0
블루 LED작동남은시간     0
블루 LED동작강도       0
카메라상태            0
냉방온도             0
난방온도             0
기준온도             0
난방부하             0
냉방부하             0
총추정광량            0
백색광추정광량          0
적색광추정광량          0
청색광추정광량          0
dtype: int64

In [ ]:
# --------------------------------------------------
# Combine Target
# --------------------------------------------------

daily_features["target"] = list(target_df[0])
daily_features

,내부온도관측치,내부습도관측치,CO2관측치,EC관측치,외부온도관측치,외부습도관측치,펌프상태,펌프작동남은시간,최근분무량,일간누적분무량,...,냉방온도,난방온도,기준온도,난방부하,냉방부하,총추정광량,백색광추정광량,적색광추정광량,청색광추정광량,target
0,22.610191,27.499558,390.902173,0.914892,19.152499,10.031938,0.574846,0.381203,169.545101,463.757065,...,22.395225,20.395225,21.395372,0.000000,1.170347,0.000000,0.000000,0.000000,0.000000,0.500000
1,22.026406,32.824389,366.103499,0.908451,17.760317,10.943132,2.632986,1.570721,723.045503,6012.706977,...,22.395832,20.395832,21.395832,0.000000,0.049771,0.000000,0.000000,0.000000,0.000000,0.611113
2,21.913938,33.985905,356.089081,0.906480,18.085376,13.975654,2.462675,1.393712,710.742596,5890.533814,...,22.395837,20.395837,21.395837,0.000000,0.163571,0.000000,0.000000,0.000000,0.000000,0.604763
3,22.907583,27.138213,347.965903,0.909050,19.912827,20.390359,2.355230,1.431507,761.830619,6114.874233,...,22.395837,20.395837,21.395837,0.000000,1.967777,0.000000,0.000000,0.000000,0.000000,0.215556
4,23.560330,39.059592,377.087964,0.910128,21.987120,21.837814,2.350764,1.388550,718.589213,5887.629565,...,22.395824,20.395824,21.395824,0.000000,4.234073,0.000000,0.000000,0.000000,0.000000,0.241235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20,24.636111,72.244135,496.022187,2.015054,24.413956,35.463861,3.657301,5.331875,498.804385,4989.097791,...,21.793054,19.857345,20.540426,1.403303,24.757463,194.577217,140.942005,28.094750,24.324229,-0.049569
21,24.614787,73.126796,506.382145,1.950773,24.498019,37.836325,3.627331,3.748520,503.994860,4412.305747,...,21.858645,19.915950,20.601621,1.491274,24.993914,196.674652,140.802266,27.668258,25.284997,-0.103974
22,24.439735,72.209942,493.126579,1.847960,24.222525,37.336357,3.900290,9.673614,4879.098209,8638.352355,...,21.712910,19.755849,20.425072,1.433873,24.910251,188.330396,135.130137,27.390676,24.399127,-0.288857
23,24.091029,71.089869,486.005183,1.660984,24.089231,37.159197,3.755142,6.445491,1487.479155,11852.071872,...,21.763354,19.803530,20.483821,1.829842,23.650880,188.984916,135.936525,28.327482,24.856924,-0.403264


In [ ]:
# --------------------------------------------------
# Feature Selection (Correlation Analysis)
# --------------------------------------------------

corr = daily_features.corr()["target"]
corr = pd.DataFrame(corr.sort_values(ascending = False), columns = ['target'])
corr.style.background_gradient(cmap='viridis')

,target
target,1.000000
기준온도,0.344553
난방작동남은시간,0.334024
내부온도관측치,0.263516
백색광추정광량,0.181827
난방온도,0.159716
CO2관측치,0.091121
화이트 LED작동남은시간,0.051491
레드 LED작동남은시간,0.050829
난방상태,0.042903


In [ ]:
selected_features = corr[abs(corr) >= 0.15].index.tolist()

selected_features.remove("target")

X = daily_features[selected_features]
y = target_df

In [ ]:
# --------------------------------------------------
# Model Training (XGBoost + KFold)
# --------------------------------------------------

kfold = KFold(n_splits=5, shuffle=True, random_state=123)

models = []

for train_idx, val_idx in kfold.split(X):

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        max_depth=6,
        n_estimators=10000,
        learning_rate=0.009,
        subsample=1,
        colsample_bytree=0.9,
        seed=2022
    )

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model.fit(X_train, y_train)

    models.append(model)

[0]	validation_0-rmse:0.327609
Will train until validation_0-rmse hasn't improved in 100 rounds.
[1]	validation_0-rmse:0.324976
[2]	validation_0-rmse:0.32242
[3]	validation_0-rmse:0.319836
[4]	validation_0-rmse:0.317278
[5]	validation_0-rmse:0.314744
[6]	validation_0-rmse:0.312241
[7]	validation_0-rmse:0.309763
[8]	validation_0-rmse:0.307321
[9]	validation_0-rmse:0.304895
[10]	validation_0-rmse:0.302473
[11]	validation_0-rmse:0.300077
[12]	validation_0-rmse:0.297736
[13]	validation_0-rmse:0.29544
[14]	validation_0-rmse:0.293201
[15]	validation_0-rmse:0.290918
[16]	validation_0-rmse:0.288699
[17]	validation_0-rmse:0.286472
[18]	validation_0-rmse:0.284249
[19]	validation_0-rmse:0.282048
[20]	validation_0-rmse:0.279842
[21]	validation_0-rmse:0.277705
[22]	validation_0-rmse:0.275607
[23]	validation_0-rmse:0.273532
[24]	validation_0-rmse:0.271474
[25]	validation_0-rmse:0.269428
[26]	validation_0-rmse:0.267376
[27]	validation_0-rmse:0.265364
[28]	validation_0-rmse:0.26335
[29]	validation_0-r

In [ ]:
# --------------------------------------------------
# Test Prediction
# --------------------------------------------------

predictions = []

for case in range(len(test_input_list)):

    test = pd.read_csv(test_input_list[case])
    test.columns = columns

    total_days = int(len(test)) // 1440
    day = 1

    case_features = []

    while day <= total_days:

        minute_data = test.iloc[1440*(day-1):1440*day]

        feature_vector = []

        for col in columns[1:]:

            if (
                "상태" not in col
                and "남은시간" not in col
                and "분무량" not in col
            ):
                minute_data[col] = remove_outliers_iqr(minute_data[col])

            feature_vector.append(minute_data[col].mean())

        case_features.append(feature_vector)

        day += 1

    case_features = pd.DataFrame(case_features)
    case_features.columns = columns[1:]

    case_features = case_features[selected_features]

    # ensemble prediction
    preds = np.mean([model.predict(case_features) for model in models], axis=0)
    print(preds)
    predictions.append(preds)

[array([0.26114833, 0.2975627 , 0.29929292, 0.3026944 , 0.3035521 ,
       0.25903642, 0.28855705, 0.29341766, 0.29061198, 0.28840107,
       0.28923896, 0.2994327 , 0.30336356, 0.30870354, 0.2937299 ,
       0.27157533, 0.29584205, 0.2960998 , 0.280517  , 0.29676765,
       0.3047364 , 0.29608795, 0.29007858, 0.38177916, 0.32856777,
       0.28429228, 0.2854311 , 0.2562367 , 0.2959417 ], dtype=float32), array([0.2810313 , 0.31625497, 0.30815277, 0.3368952 , 0.31360596,
       0.27436438, 0.2738693 , 0.27964064, 0.3057832 , 0.28327918,
       0.27147204, 0.30195913, 0.29578775, 0.31085318, 0.29690942,
       0.3353696 , 0.28994125, 0.2893563 , 0.29402697, 0.30567336,
       0.31002122, 0.31510633, 0.28707606, 0.2944653 , 0.30347863,
       0.31199396, 0.3137966 , 0.28914428, 0.31401956], dtype=float32), array([0.27556455, 0.28425145, 0.28647947, 0.26481906, 0.28000006,
       0.2825515 , 0.31885535, 0.3105199 , 0.32008457, 0.32026052,
       0.31806654, 0.3092475 , 0.29916131, 0.310474